In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, confusion_matrix

In [ ]:
data = pd.read_csv('Student_performance_10k.csv')
print('Shape:', data.shape)
data.head()

In [ ]:
print('=== Before Cleaning ===')
print('Missing values per column:')
print(data.isnull().sum())
print('\nTotal missing:', data.isnull().sum().sum())
print('Duplicates   :', data.duplicated().sum())

# Fix mixed-type columns: convert score columns to numeric (some rows have tab characters)
score_cols = ['math_score', 'reading_score', 'writing_score', 'science_score', 'total_score']
for col in score_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

# Drop rows with any remaining missing values
data.dropna(inplace=True)

# Drop non-informative ID column
data.drop(columns=['roll_no'], inplace=True)

print('\n=== After Cleaning ===')
print('Shape:', data.shape)
print('Missing values:', data.isnull().sum().sum())

In [ ]:
print(data['grade'].value_counts().sort_index())

counts = data['grade'].value_counts().sort_index()
plt.figure(figsize=(7, 4))
palette = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']
bars = plt.bar(counts.index, counts.values, color=palette[:len(counts)], edgecolor='white', width=0.6)
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             str(val), ha='center', fontweight='bold', fontsize=10)
plt.title('Grade Distribution in Dataset', fontsize=13, fontweight='bold')
plt.xlabel('Grade'); plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
score_features = ['math_score', 'reading_score', 'writing_score', 'science_score']

print('Shape before outlier removal:', data.shape)
for col in score_features:
    Q1, Q3 = data[col].quantile(0.25), data[col].quantile(0.75)
    IQR = Q3 - Q1
    data = data[(data[col] >= Q1 - 3*IQR) & (data[col] <= Q3 + 3*IQR)]

print('Shape after  outlier removal:', data.shape)

In [ ]:
# Encode categorical features
le_gender = LabelEncoder()
le_race   = LabelEncoder()
le_edu    = LabelEncoder()

data['gender']                      = le_gender.fit_transform(data['gender'])
data['race_ethnicity']              = le_race.fit_transform(data['race_ethnicity'])
data['parental_level_of_education'] = le_edu.fit_transform(data['parental_level_of_education'])

# Encode target label
le = LabelEncoder()
data['grade'] = le.fit_transform(data['grade'])
print('Target classes:', le.classes_)

# Feature Selection
# Drop total_score: it equals math+reading+writing+science → data leakage
X = data.drop(columns=['grade', 'total_score'])
y = data['grade']

print('\nSelected features:', X.columns.tolist())
print('X shape:', X.shape, '| y shape:', y.shape)

In [ ]:
# ── Data Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=23, stratify=y
)
print('Train set:', X_train.shape)
print('Test  set:', X_test.shape)

In [ ]:
# ── Data Scaling ──
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
print('StandardScaler applied. Mean ≈ 0, Std ≈ 1 on training set.')

In [ ]:
# ── Algorithm 1: Decision Tree ──
dt = DecisionTreeClassifier(random_state=23)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

dt_acc  = accuracy_score(y_test, y_pred_dt)
dt_prec = precision_score(y_test, y_pred_dt, average='weighted')
dt_rec  = recall_score(y_test, y_pred_dt, average='weighted')

print('=== Decision Tree ===')
print(f'Accuracy : {dt_acc:.4f}')
print(f'Precision: {dt_prec:.4f}')
print(f'Recall   : {dt_rec:.4f}')
print(classification_report(y_test, y_pred_dt, target_names=le.classes_))

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix – Decision Tree')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout(); plt.show()

In [ ]:
# ── Algorithm 2: Random Forest ──
rf = RandomForestClassifier(n_estimators=100, random_state=23)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rf_acc  = accuracy_score(y_test, y_pred_rf)
rf_prec = precision_score(y_test, y_pred_rf, average='weighted')
rf_rec  = recall_score(y_test, y_pred_rf, average='weighted')

print('=== Random Forest ===')
print(f'Accuracy : {rf_acc:.4f}')
print(f'Precision: {rf_prec:.4f}')
print(f'Recall   : {rf_rec:.4f}')
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix – Random Forest')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout(); plt.show()

# Feature Importances
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(8, 4))
importances.plot(kind='barh', color='#27ae60')
plt.title('Feature Importances – Random Forest')
plt.tight_layout(); plt.show()

In [ ]:
# ── Algorithm 3: KNN ──
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

knn_acc  = accuracy_score(y_test, y_pred_knn)
knn_prec = precision_score(y_test, y_pred_knn, average='weighted')
knn_rec  = recall_score(y_test, y_pred_knn, average='weighted')

print('=== KNN (K=5) ===')
print(f'Accuracy : {knn_acc:.4f}')
print(f'Precision: {knn_prec:.4f}')
print(f'Recall   : {knn_rec:.4f}')
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_knn), annot=True, fmt='d', cmap='Oranges',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix – KNN')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout(); plt.show()

In [ ]:
# ── Conclusion: Comparison ──
results_df = pd.DataFrame({
    'Algorithm': ['Decision Tree', 'Random Forest', 'KNN'],
    'Accuracy' : [dt_acc,  rf_acc,  knn_acc],
    'Precision': [dt_prec, rf_prec, knn_prec],
    'Recall'   : [dt_rec,  rf_rec,  knn_rec]
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print(results_df.to_string(index=False))

x = np.arange(3)
w = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, results_df['Accuracy'],  w, label='Accuracy',  color='#3498db')
ax.bar(x,     results_df['Precision'], w, label='Precision', color='#2ecc71')
ax.bar(x + w, results_df['Recall'],    w, label='Recall',    color='#e67e22')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Algorithm'])
ax.set_ylim(0, 1.15)
ax.set_title('Algorithms Comparison – Student Performance Grade Prediction', fontsize=13)
ax.legend()
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
plt.tight_layout()
plt.show()

best = results_df.iloc[0]
print(f"\n✅ Best Model : {best['Algorithm']}")
print(f"   Accuracy   : {best['Accuracy']:.4f}")
print(f"   Precision  : {best['Precision']:.4f}")
print(f"   Recall     : {best['Recall']:.4f}")